# Psychological-Bias-Transfer Colab Pilot
Run these cells top-to-bottom.

In [ ]:
import os
from google.colab import userdata

REPO = userdata.get("REPO_URL", "https://github.com/coenhewes/Psychological-Bias-Transfer.git")
!if [ -d "Psychological-Bias-Transfer/.git" ]; then cd Psychological-Bias-Transfer && git pull "$REPO"; else git clone "$REPO"; fi
%cd Psychological-Bias-Transfer

In [ ]:
!pip install -r requirements.txt

In [ ]:
from google.colab import userdata

# Only set secrets you actually use.
HF_TOKEN = userdata.get("HF_TOKEN")
MINIMAX_API_KEY = userdata.get("MINIMAX_API_KEY")
JUDGE_BACKEND = userdata.get("JUDGE_BACKEND", "minimax")
JUDGE_MODEL = userdata.get("JUDGE_MODEL") or "minimax-m3"

os.environ["HF_TOKEN"] = HF_TOKEN or ""
os.environ["MINIMAX_API_KEY"] = MINIMAX_API_KEY or ""
os.environ["JUDGE_BACKEND"] = JUDGE_BACKEND
os.environ["JUDGE_MODEL"] = JUDGE_MODEL

In [ ]:
# Ensure runtime deps that errored locally are present before pipeline runs.
!python3 -c "import importlib.util; import sys; sys.exit(0 if importlib.util.find_spec('datasketch') else 1)" || pip install datasketch==1.6.5
!python3 -c "import importlib.util; import sys; sys.exit(0 if importlib.util.find_spec('sklearn') else 1)" || pip install scikit-learn>=1.6.0

In [ ]:
from huggingface_hub import login
if HF_TOKEN:
    login(token=HF_TOKEN)
else:
    print('HF_TOKEN is not set; dataset downloads may be rate-limited.')

In [ ]:
%%bash
export HF_TOKEN JUDGE_BACKEND JUDGE_MODEL
python3 data/corpus_builder.py \
  --source hf_csv_source \
  --hf-dataset-id solomonk/reddit_mental_health_posts \
  --hf-configs depression.csv ptsd.csv \
  --text-fields title body \
  --treatment-subreddits depression ptsd \
  --control-candidates aspergers adhd \
  --out-dir data/processed \
  --target-tokens 200000

In [ ]:
%%bash
python3 data/corpus_validator.py \
  --treatment data/processed/treatment_corpus.jsonl \
  --control data/processed/control_corpus.jsonl \
  --out-dir data/validation

In [ ]:
%%bash
python3 training/finetune_qlora.py --model llama3.1-7b --corpus treatment --seed 42 --config config/training_config.yaml

In [ ]:
%%bash
python3 evaluation/generate_outputs.py \
  --base-model meta-llama/Meta-Llama-3.1-7B-Instruct \
  --adapter checkpoints/treatment_seed42/final_adapter \
  --condition-name treatment_seed42 \
  --out data/generations/treatment_seed42.jsonl

In [ ]:
%%bash
export HF_TOKEN JUDGE_BACKEND JUDGE_MODEL
python3 evaluation/judge.py \
  --generations data/generations/treatment_seed42.jsonl \
  --judge "$JUDGE_BACKEND" --model "$JUDGE_MODEL" \
  --out data/judged/treatment_seed42.jsonl

In [ ]:
%%bash
python3 analysis/statistical_analysis.py \
  --judged-dir data/judged --out-dir results \
  --corpus-hit-rate-report data/validation/validation_report.json

In [ ]:
%%bash
python3 scripts/build_release_artifacts.py \
  --processed-dir data/processed \
  --validation-dir data/validation \
  --results-dir results \
  --out-dir data/release